# 선데이메이플 보상 패턴 분석 · 다음 선데이는 무엇인가



## 1. 데이터 준비

인벤 보관본을 재크롤링 없이 로드하고, 기존 `src` 함수로 정규식 파싱·보상명 정규화·주기 계산을 수행해 14컬럼 tidy 데이터를 메모리에서 만든다. 분석 대상은 미래 빈 블록과 보상 없음 행을 제외한 1118행이다.

In [1]:
from pathlib import Path
import json
import logging
import re
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.features import compute_cycle_days, merge_is_discontinued
from src.normalize import categorize, extract_magnitude, normalize_reward
from src.parse import parse_by_date, parse_by_type
from src.predict_dataset import DEFAULT_RECENCY_GATE_DAYS, DOMAIN_DISCONTINUED, _candidate_rewards, _eligible_rows, build_inference_frame, build_prediction_frame, gate_candidates
from src.prophet_cycle import build_weekly_series, fit_predict_reward, rank_candidates

warnings.filterwarnings("ignore", message="IProgress not found.*")
logging.getLogger("cmdstanpy").disabled = True
logging.getLogger("prophet").disabled = True
logging.getLogger("prophet.plot").disabled = True

TODAY = pd.Timestamp("2026-06-10")
SPLIT_CUTOFF = pd.Timestamp("2025-01-01")
RANDOM_SEED = 42
raw_dir = ROOT / "data" / "raw"
manual_obs_path = ROOT / "data" / "interim" / "manual_obs_20260614.csv"
meta = json.loads((raw_dir / "inven_20563_meta.json").read_text(encoding="utf-8"))
by_type_text = (raw_dir / "view_by_type.txt").read_text(encoding="utf-8")
by_date_text = (raw_dir / "view_by_date.txt").read_text(encoding="utf-8")
source_summary = {
    "출처": meta["url"],
    "수집시각": meta["fetched_at"],
    "종류별_문자수": len(by_type_text),
    "날짜별_문자수": len(by_date_text),
}
display(source_summary)
display(pd.DataFrame({"종류별_앞부분": by_type_text.splitlines()[:8]}))
display(pd.DataFrame({"날짜별_앞부분": by_date_text.splitlines()[:8]}))

{'출처': 'https://www.inven.co.kr/board/maple/2304/20563',
 '수집시각': '2026-06-10T06:08:50.763291+00:00',
 '종류별_문자수': 23476,
 '날짜별_문자수': 43800}

,종류별_앞부분
0,"- [HEXA 스탯] 메인 스탯이 5레벨 이상일 때, 메인 스탯의 강화 확률 20% 증가"
1,2024.07.21
2,(+84일)2024.10.13
3,(+91일)2025.01.12
4,(+84일)2025.04.06
5,(+91일)2025.07.06
6,(+112일)2025.10.26
7,(+77일)2026.01.11


,날짜별_앞부분
0,2017.03.12 (-)
1,주문의 흔적 강화 비용 30% 할인
2,"(리부트, 리부트2월드 : 하루 종일 드롭률 2배)"
3,돌발미션 일일 클리어 가능 횟수 2배(3회→6회)
4,돌발미션 클리어 경험치 2배
5,[[S]]2017.03.19[[/S]]
6,[[S]]아예 없음[[/S]]
7,2017.03.26 (도전! 메이플 레인보우!)


### 전처리

두 원시 뷰를 기존 `src` 파서로 정규식 파싱하고, 보상명 정규화·카테고리 매핑·재등장 주기·단종 여부를 붙여 14컬럼 tidy 데이터를 만든다. 관측 단위는 `(date, reward_raw)`이며, 이 노트북은 결과를 `data/processed`에 저장하지 않는다.

In [2]:
by_type_records = parse_by_type(by_type_text)
by_date_records = parse_by_date(by_date_text)
parsed_by_type = pd.DataFrame(
    by_type_records,
    columns=["reward_raw", "date", "stated_cycle_days", "is_discontinued", "trailing"],
)
parsed_by_date = pd.DataFrame(
    by_date_records,
    columns=["date", "event_context", "is_major_event", "no_event_week", "reward_raw", "reward_trailing"],
)

def is_special(row):
    text = " ".join(
        str(x)
        for x in [row.get("reward_raw"), row.get("reward_trailing"), row.get("event_context")]
        if pd.notna(x)
    )
    return "타임" in text

def time_limited(trailing):
    if pd.isna(trailing) or trailing is None:
        return None
    text = str(trailing)
    if "하루 종일" in text or "하루종일" in text:
        return "하루 종일"
    if "오전" in text or "오후" in text or "~" in text:
        return text
    if re.search(r"\d+분", text):
        return text
    return None

tidy_prenorm = parsed_by_date.copy()
tidy_prenorm["is_special"] = tidy_prenorm.apply(is_special, axis=1)
tidy_prenorm["time_limited"] = tidy_prenorm["reward_trailing"].apply(time_limited)
tidy_prenorm["is_future"] = pd.to_datetime(tidy_prenorm["date"]) > TODAY
tidy_prenorm = tidy_prenorm[
    [
        "date",
        "reward_raw",
        "reward_trailing",
        "event_context",
        "is_major_event",
        "is_special",
        "time_limited",
        "is_future",
        "no_event_week",
    ]
]

tidy = tidy_prenorm.copy()
tidy["reward_norm"] = tidy["reward_raw"].apply(lambda value: normalize_reward(str(value)) if pd.notna(value) else None)
tidy["category"] = tidy["reward_norm"].apply(lambda value: categorize(value) if pd.notna(value) else None)
tidy["reward_magnitude"] = tidy["reward_raw"].apply(lambda value: extract_magnitude(str(value)) if pd.notna(value) else None)
tidy["cycle_days"] = compute_cycle_days(tidy)
tidy["is_discontinued"] = merge_is_discontinued(tidy, parsed_by_type)
tidy = tidy[
    [
        "date",
        "reward_raw",
        "reward_trailing",
        "reward_norm",
        "category",
        "reward_magnitude",
        "event_context",
        "is_major_event",
        "is_special",
        "time_limited",
        "is_discontinued",
        "is_future",
        "no_event_week",
        "cycle_days",
    ]
]
tidy["date"] = pd.to_datetime(tidy["date"])
analysis = tidy.loc[tidy["reward_norm"].notna() & ~tidy["is_future"].astype(bool)].copy()
display(tidy.head())

,date,reward_raw,reward_trailing,reward_norm,category,reward_magnitude,event_context,is_major_event,is_special,time_limited,is_discontinued,is_future,no_event_week,cycle_days
0,2017-03-12,주문의 흔적 강화 비용 30% 할인,NaN,주문의흔적_비용할인,강화,30%,NaN,False,False,NaN,True,False,False,<NA>
1,2017-03-12,"(리부트, 리부트2월드 : 하루 종일 드롭률 2배)",NaN,리부트_드롭률,수집기타,2배,NaN,False,False,NaN,True,False,False,<NA>
2,2017-03-12,돌발미션 일일 클리어 가능 횟수 2배,(3회→6회),돌발미션,사냥효율,2배,NaN,False,False,NaN,True,False,False,<NA>
3,2017-03-12,돌발미션 클리어 경험치 2배,NaN,돌발미션,사냥효율,2배,NaN,False,False,NaN,True,False,False,<NA>
4,2017-03-19,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN,<NA>,False,True,<NA>


In [3]:
non_sunday_rows = tidy.loc[tidy["date"].dt.weekday != 6, ["date", "reward_norm", "event_context"]].copy()
non_sunday_rows["date"] = non_sunday_rows["date"].dt.strftime("%Y-%m-%d")
quality_summary = {
    "total_rows": len(tidy),
    "analysis_rows": len(analysis),
    "columns": len(tidy.columns),
    "unique_reward_norm": analysis["reward_norm"].nunique(),
    "discontinued_unique_reward_raw": parsed_by_type.loc[parsed_by_type["is_discontinued"], "reward_raw"].nunique(),
    "date_reward_raw_duplicates": int(tidy.loc[tidy["reward_raw"].notna()].duplicated(["date", "reward_raw"]).sum()),
    "non_sunday_rows": len(non_sunday_rows),
}
assert quality_summary["total_rows"] == 1160
assert quality_summary["analysis_rows"] == 1118
assert quality_summary["columns"] == 14
assert quality_summary["discontinued_unique_reward_raw"] == 71
assert quality_summary["date_reward_raw_duplicates"] == 0
assert quality_summary["non_sunday_rows"] == 11
summary_labels = {
    "total_rows": "전체행",
    "analysis_rows": "분석행",
    "columns": "컬럼수",
    "unique_reward_norm": "고유_reward_norm",
    "discontinued_unique_reward_raw": "단종_고유_reward_raw",
    "date_reward_raw_duplicates": "date_reward_raw_중복",
    "non_sunday_rows": "비일요일_행",
}
display(pd.Series(quality_summary).rename(index=summary_labels).to_frame("값"))
display(non_sunday_rows.reset_index(drop=True))

,값
전체행,1160
분석행,1118
컬럼수,14
고유_reward_norm,48
단종_고유_reward_raw,71
date_reward_raw_중복,0
비일요일_행,11


,date,reward_norm,event_context
0,2023-01-12,스타포스_1+1강화,요정 웡키의 웡스토랑 - 마이크의 파워업 위크
1,2023-01-13,위대한소울_확률,요정 웡키의 웡스토랑 - 마이크의 파워업 위크
2,2023-01-14,스타포스_성공확률,요정 웡키의 웡스토랑 - 마이크의 파워업 위크
3,2023-01-14,스타포스_비용할인,요정 웡키의 웡스토랑 - 마이크의 파워업 위크
4,2023-01-16,주문의흔적_비용할인,요정 웡키의 웡스토랑 - 마이크의 파워업 위크
5,2023-01-17,어빌리티_재설정할인,요정 웡키의 웡스토랑 - 마이크의 파워업 위크
6,2023-01-18,스타포스_비용할인,요정 웡키의 웡스토랑 - 마이크의 파워업 위크
7,2024-04-01,선물_기타지급,유니온 릴레이
8,2024-04-01,현상금_경험치,유니온 릴레이
9,2024-04-01,불꽃늑대_경험치,유니온 릴레이


## 2. 다음 선데이 예측

분류기는 경과일·과거 출현율·최근 재등장 주기 피처를 사용하고, Prophet은 보상별 주간 0/1 시계열의 28일 주기와 연간 변동을 사용한다. 두 방법은 같은 후보와 같은 `date_hit_rate`로 비교한다. 이 셀은 Prophet 후보 20개를 적합하므로 실행에 수십 초가 걸릴 수 있다.

06-14 검증에서는 게이트 전 예측 상위 5개가 실제 보상을 하나도 포함하지 못했고, 게이트 후에는 몬스터파크_경험치가 5위로 올라와 1/2를 맞혔다. 선물_기타지급은 두 경우 모두 후보권 밖이다.

In [6]:
prediction_frame = build_prediction_frame(tidy, candidate_cutoff=SPLIT_CUTOFF)
train = prediction_frame.loc[prediction_frame["date"] < SPLIT_CUTOFF].copy()
test = prediction_frame.loc[prediction_frame["date"] >= SPLIT_CUTOFF].copy()
feature_columns = [
    "days_since_last",
    "past_count",
    "past_rate",
    "roll_gap_mean",
    "roll_gap_median",
    "lag1_appeared",
    "category",
    "month",
    "quarter",
    "weekofyear",
]

def evaluate(y_true, y_score, dates):
    y_true = pd.Series(y_true).astype(int).reset_index(drop=True)
    y_score = pd.Series(y_score).astype(float).reset_index(drop=True)
    dates = pd.Series(dates).reset_index(drop=True)
    y_pred = (y_score >= 0.5).astype(int)
    work = pd.DataFrame({"date": dates, "y": y_true, "score": y_score})
    def date_hit(group):
        k = int(group["y"].sum())
        if k < 1:
            return np.nan
        return float(group.nlargest(k, "score")["y"].max())
    auc = roc_auc_score(y_true, y_score) if y_true.nunique() > 1 else np.nan
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": auc,
        "date_hit_rate": work.groupby("date", sort=True).apply(date_hit).mean(),
    }

X_all = pd.get_dummies(prediction_frame[feature_columns], columns=["category"], dummy_na=True).fillna(-1)
X_train = X_all.loc[train.index]
X_test = X_all.loc[test.index]
y_train = train["y"].astype(int)
y_test = test["y"].astype(int)
b1_score = (test["roll_gap_median"].notna() & (test["days_since_last"] >= test["roll_gap_median"])).astype(float)
b2_score = test["past_rate"].clip(0, 1)
logistic_model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_SEED)
tree_model = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=RANDOM_SEED)
logistic_model.fit(X_train, y_train)
tree_model.fit(X_train, y_train)
logistic_score = logistic_model.predict_proba(X_test)[:, 1]
tree_score = tree_model.predict_proba(X_test)[:, 1]
classifier_results = pd.DataFrame(
    [
        {"방법": "간격규칙", **evaluate(y_test, b1_score, test["date"])},
        {"방법": "사전확률", **evaluate(y_test, b2_score, test["date"])},
        {"방법": "로지스틱", **evaluate(y_test, logistic_score, test["date"])},
        {"방법": "랜덤포레스트", **evaluate(y_test, tree_score, test["date"])},
    ]
)

eligible_for_prophet = _eligible_rows(tidy)
candidates = _candidate_rewards(eligible_for_prophet.loc[eligible_for_prophet["date"] < SPLIT_CUTOFF], 20)
test_dates = pd.Index(sorted(eligible_for_prophet.loc[eligible_for_prophet["date"] >= SPLIT_CUTOFF, "date"].drop_duplicates()))
actual_by_date = {date: set(group["reward_norm"]) for date, group in eligible_for_prophet.groupby("date", sort=False)}
prophet_scores = {}
for reward in candidates:
    series = build_weekly_series(tidy, reward, SPLIT_CUTOFF)
    forecast = fit_predict_reward(series, pd.DatetimeIndex(test_dates))
    prophet_scores[reward] = forecast.set_index("ds")["yhat"]
prophet_hits = []
for date in test_dates:
    gated = gate_candidates(eligible_for_prophet, candidates, date, recency_gate_days=DEFAULT_RECENCY_GATE_DAYS, domain_discontinued=DOMAIN_DISCONTINUED)
    actual = actual_by_date.get(date, set())
    k = len(actual & set(candidates))
    if k < 1 or not gated:
        prophet_hits.append(np.nan)
    else:
        ranked = sorted(gated, key=lambda reward: (-float(prophet_scores[reward].loc[date]), reward))
        prophet_hits.append(float(bool(set(ranked[:k]) & actual)))
prophet_date_hit_rate = pd.Series(prophet_hits).mean()
method_compare = pd.DataFrame(
    [
        {"방법": "간격규칙", "date_hit_rate": float(classifier_results.loc[classifier_results["방법"] == "간격규칙", "date_hit_rate"].iloc[0])},
        {"방법": "사전확률", "date_hit_rate": float(classifier_results.loc[classifier_results["방법"] == "사전확률", "date_hit_rate"].iloc[0])},
        {"방법": "랜덤포레스트", "date_hit_rate": float(classifier_results.loc[classifier_results["방법"] == "랜덤포레스트", "date_hit_rate"].iloc[0])},
        {"방법": "Prophet", "date_hit_rate": prophet_date_hit_rate},
    ]
)
assert round(float(method_compare.loc[method_compare["방법"] == "랜덤포레스트", "date_hit_rate"].iloc[0]), 4) == 0.4493
assert round(float(method_compare.loc[method_compare["방법"] == "Prophet", "date_hit_rate"].iloc[0]), 4) == 0.4203
display(classifier_results)
display(method_compare)

,방법,accuracy,f1,roc_auc,date_hit_rate
0,간격규칙,0.416000,0.200730,0.520324,0.347826
1,사전확률,0.888000,0.000000,0.700723,0.362319
2,로지스틱,0.532000,0.298000,0.748534,0.391304
3,랜덤포레스트,0.860667,0.125523,0.734951,0.449275


,방법,date_hit_rate
0,간격규칙,0.347826
1,사전확률,0.362319
2,랜덤포레스트,0.449275
3,Prophet,0.420290


In [7]:
manual_obs = pd.read_csv(manual_obs_path, parse_dates=["date"])
actual_0614 = set(manual_obs["reward_norm"])
monster_park = "\ubaac\uc2a4\ud130\ud30c\ud06c_\uacbd\ud5d8\uce58"

label_map = {
    "group": "\uad6c\ubd84",
    "top5_hits": "\uc0c1\uc7045_\uc801\uc911\uc218",
    "candidate_count": "\ud6c4\ubcf4\uc218",
    "reward_norm": "reward_norm",
    "included": "\ud6c4\ubcf4\ud3ec\ud568",
    "rank": "\uc21c\uc704",
    "score_pct": "\uc810\uc218(%)",
    "predicted_reward": "\uc608\uce21\ubcf4\uc0c1",
    "actual_hit": "\uc2e4\uc81c\ub4f1\uc7a5",
    "actual_reward": "\uc2e4\uc81c_06_14_\ubcf4\uc0c1",
}

def predict_classifier(tidy_source, target_date, gate):
    infer_frame = build_inference_frame(
        tidy_source,
        target_date,
        candidate_cutoff=SPLIT_CUTOFF,
        recency_gate_days=DEFAULT_RECENCY_GATE_DAYS if gate else None,
        domain_discontinued=DOMAIN_DISCONTINUED if gate else frozenset(),
    )
    X_target = pd.get_dummies(infer_frame[feature_columns], columns=["category"], dummy_na=True)
    X_target = X_target.reindex(columns=X_all.columns, fill_value=0).fillna(-1)
    table = infer_frame[["date", "reward_norm"]].copy()
    table["score"] = tree_model.predict_proba(X_target)[:, 1]
    table = table.sort_values(["score", "reward_norm"], ascending=[False, True]).reset_index(drop=True)
    table["rank"] = np.arange(1, len(table) + 1)
    table["score_pct"] = (table["score"] * 100).round(1)
    return table

def actual_rank(table, actual_rewards):
    rows = []
    for reward in sorted(actual_rewards):
        matched = table.loc[table["reward_norm"] == reward]
        if matched.empty:
            rows.append({"reward_norm": reward, "included": False, "rank": pd.NA, "score_pct": pd.NA})
        else:
            row = matched.iloc[0]
            rows.append({"reward_norm": reward, "included": True, "rank": int(row["rank"]), "score_pct": float(row["score_pct"])})
    return pd.DataFrame(rows)

def top5_with_actual(table, actual_rewards):
    view = table.head(5)[["rank", "reward_norm", "score_pct"]].copy()
    view["actual_hit"] = view["reward_norm"].isin(actual_rewards).map({True: "O", False: "X"})
    return view.rename(columns={"reward_norm": "predicted_reward"})

pred_0614_raw = predict_classifier(tidy, pd.Timestamp("2026-06-14"), gate=False)
pred_0614_gate = predict_classifier(tidy, pd.Timestamp("2026-06-14"), gate=True)
gate_summary = pd.DataFrame(
    [
        {"group": "\uac8c\uc774\ud2b8 \uc804", "top5_hits": int(pred_0614_raw.head(5)["reward_norm"].isin(actual_0614).sum()), "candidate_count": len(pred_0614_raw)},
        {"group": "\uac8c\uc774\ud2b8 \ud6c4", "top5_hits": int(pred_0614_gate.head(5)["reward_norm"].isin(actual_0614).sum()), "candidate_count": len(pred_0614_gate)},
    ]
)
rank_raw = actual_rank(pred_0614_raw, actual_0614).assign(**{"group": "\uac8c\uc774\ud2b8 \uc804"})
rank_gate = actual_rank(pred_0614_gate, actual_0614).assign(**{"group": "\uac8c\uc774\ud2b8 \ud6c4"})
raw_top5_actual = top5_with_actual(pred_0614_raw, actual_0614)
gate_top5_actual = top5_with_actual(pred_0614_gate, actual_0614)
prophet_0614 = rank_candidates(tidy, candidates, as_of=pd.Timestamp("2026-06-14"), target_date=pd.Timestamp("2026-06-14"), recency_gate_days=DEFAULT_RECENCY_GATE_DAYS, domain_discontinued=DOMAIN_DISCONTINUED)
prophet_0614["yhat"] = prophet_0614["yhat"].round(3)
assert gate_summary.loc[0, "top5_hits"] == 0
assert gate_summary.loc[1, "top5_hits"] == 1
assert int(rank_raw.loc[rank_raw["reward_norm"] == monster_park, "rank"].iloc[0]) == 9
assert int(rank_gate.loc[rank_gate["reward_norm"] == monster_park, "rank"].iloc[0]) == 5
assert raw_top5_actual["actual_hit"].eq("O").sum() == 0
assert gate_top5_actual.loc[gate_top5_actual["predicted_reward"] == monster_park, "actual_hit"].iloc[0] == "O"
assert int(gate_top5_actual.loc[gate_top5_actual["predicted_reward"] == monster_park, "rank"].iloc[0]) == 5
display(pd.DataFrame({"actual_reward": sorted(actual_0614)}).rename(columns=label_map))
display(gate_summary.rename(columns=label_map))
display(pd.concat([rank_raw, rank_gate], ignore_index=True)[["group", "reward_norm", "included", "rank", "score_pct"]].rename(columns=label_map))
display(raw_top5_actual.rename(columns=label_map))
display(gate_top5_actual.rename(columns=label_map))
display(prophet_0614.head(5))

,구분,상위5_적중수,후보수
0,게이트 전,0,20
1,게이트 후,1,10


,구분,reward_norm,후보포함,순위,점수(%)
0,게이트 전,몬스터파크_경험치,True,9,24.0
1,게이트 전,선물_기타지급,False,<NA>,<NA>
2,게이트 후,몬스터파크_경험치,True,5,24.0
3,게이트 후,선물_기타지급,False,<NA>,<NA>


,reward_norm,yhat,rank
0,주문의흔적_비용할인,0.410,1
1,콤보킬_경험치,0.400,2
2,룬_경험치버프,0.396,3
3,몬스터파크_경험치,0.350,4
4,스타포스_비용할인,0.280,5


In [8]:
category_lookup = tidy.dropna(subset=["reward_norm", "category"]).groupby("reward_norm")["category"].agg(lambda s: s.mode().iloc[0])
manual_for_tidy = manual_obs.copy()
manual_for_tidy["category"] = manual_for_tidy["reward_norm"].map(category_lookup)
manual_for_tidy["is_future"] = False
manual_for_tidy["no_event_week"] = False
tidy_with_manual = pd.concat([tidy, manual_for_tidy], ignore_index=True, sort=False)
rf_0621 = predict_classifier(tidy_with_manual, pd.Timestamp("2026-06-21"), gate=True)
prophet_0621 = rank_candidates(tidy_with_manual, candidates, as_of=pd.Timestamp("2026-06-21"), target_date=pd.Timestamp("2026-06-21"), recency_gate_days=DEFAULT_RECENCY_GATE_DAYS, domain_discontinued=DOMAIN_DISCONTINUED)
prophet_0621["yhat"] = prophet_0621["yhat"].round(3)
forecast_compare = pd.DataFrame(
    {
        "rank": range(1, 6),
        "rf_reward": rf_0621.head(5)["reward_norm"].to_list(),
        "rf_score_pct": rf_0621.head(5)["score_pct"].to_list(),
        "prophet_reward": prophet_0621.head(5)["reward_norm"].to_list(),
        "prophet_yhat": prophet_0621.head(5)["yhat"].to_list(),
    }
)
excluded_rewards = {"\ud604\uc0c1\uae08_\uacbd\ud5d8\uce58", "\ubd88\uaf43\ub291\ub300_\uacbd\ud5d8\uce58", "\uc2a4\ud0c0\ud3ec\uc2a4_1+1\uac15\ud654", "\uc2a4\ud0c0\ud3ec\uc2a4_\uc131\uacf5\ud655\ub960"}
assert excluded_rewards.isdisjoint(set(rf_0621["reward_norm"]))
assert excluded_rewards.isdisjoint(set(prophet_0621["reward_norm"]))
assert forecast_compare.loc[0, "rf_reward"] == "\ubaac\uc2a4\ud130\uceec\ub809\uc158"
assert float(forecast_compare.loc[0, "rf_score_pct"]) == 41.7
assert forecast_compare.loc[0, "prophet_reward"] == "\ubaac\uc2a4\ud130\ud30c\ud06c_\uacbd\ud5d8\uce58"
assert float(forecast_compare.loc[0, "prophet_yhat"]) == 0.402
display(
    forecast_compare.rename(
        columns={
            "rank": "\uc21c\uc704",
            "rf_reward": "\ub79c\ub364\ud3ec\ub808\uc2a4\ud2b8_\ubcf4\uc0c1",
            "rf_score_pct": "\ub79c\ub364\ud3ec\ub808\uc2a4\ud2b8_\uc810\uc218(%)",
            "prophet_reward": "Prophet_\ubcf4\uc0c1",
            "prophet_yhat": "Prophet_yhat",
        }
    )
)

,순위,랜덤포레스트_보상,랜덤포레스트_점수(%),Prophet_보상,Prophet_yhat
0,1,몬스터컬렉션,41.7,몬스터파크_경험치,0.402
1,2,스타포스_비용할인,39.3,주문의흔적_비용할인,0.330
2,3,주문의흔적_비용할인,36.7,룬_경험치버프,0.318
3,4,어빌리티_재설정할인,34.3,콤보킬_경험치,0.281
4,5,엘리트몬스터_증가,27.7,스타포스_비용할인,0.157


## 3. 결론

선데이메이플은 한 회차에 담기는 보상 수가 늘었고, 최근으로 올수록 강화와 성장 계열이 큰 축을 이룬다. 단종은 성장·사냥효율·수집기타 계열에 집중됐다. 예측에서는 랜덤포레스트가 백테스트 `date_hit_rate` 0.4493으로 Prophet 0.4203보다 높았지만, 두 방법은 서로 다른 신호를 보며 둘 다 단종 게이트가 필요했다.